In [ ]:
from msticpy.config import MpConfigFile, MpConfigEdit

# Load existing or default settings
mpconfig = MpConfigFile()
mpconfig.load_default()
mpconfig.view_settings()

In [ ]:
from msticpy.init.user_config import load_user_defaults
load_user_defaults()

In [ ]:
from msticpy.sectools.tilookup import TILookup
from msticpy.context.geoip import GeoLiteLookup
import msticpy as mp

# Initialize with your configured providers (e.g. VirusTotal)
ti = TILookup(providers=["VirusTotal"])
# Initialize the GeoLite IP location tool
geo = GeoLiteLookup()

# Show status of loaded providers
ti.provider_status

In [ ]:
import sys
import os
import json
from pandas import Timestamp

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)
    
from Agents.AnalyzeAgent import *


In [ ]:

profiles = build_user_profiles(df)

print(f"Profile building complete. # of users: {len(profiles)}")

In [ ]:
df = split_into_conversations(df)

In [ ]:
df.tail(50)

In [ ]:
df = split_into_conversations(df)
print("Conversation split complete")

# Build all profiles
profiles = build_user_profiles(df)

# Sort them by number of messages sent, descending
sorted_profiles = dict(
    sorted(profiles.items(), key=lambda item: item[1]["messages_sent"], reverse=True)
)

# Pick top 20 most active users
limited_users = dict(list(sorted_profiles.items())[:20])

for user, profile in limited_users.items():
    for field in ["first_seen", "last_seen"]:
        if isinstance(profile.get(field), Timestamp):
            profile[field] = profile[field].isoformat()

# Save to file
with open("selected_user_profiles.json", "w", encoding="utf-8") as f:
    json.dump(limited_users, f, ensure_ascii=False, indent=2)

In [ ]:
user_messages_dict = {u: df[df["from"] == u]["translated_body"].dropna().tolist() for u in limited_users}

# run summary
PortfolioBuilder.summarize_user_messages(user_messages_dict)

In [ ]:
results = VictimAnalyzer.analyze_victims(df, llm.invoke, output_dir=".")

In [ ]:
import time
lol = time.time()
generate_ioc_tables(df)
lol2 = time.time()

print(f"time taken: {lol2-lol1} seconds")

In [ ]:
# Load selected user profiles
with open("selected_user_profiles.json", "r", encoding="utf-8") as f:
    selected_profiles = json.load(f)
    
# Enrich with IP metadata
enriched_profiles = attach_enriched_ips_to_profiles(selected_profiles, "ioc-csvs/ip_table_enriched.csv")

# Save enriched version
with open("enriched_user_profiles.json", "w", encoding="utf-8") as f:
    json.dump(enriched_profiles, f, ensure_ascii=False, indent=2, default=str)

In [ ]:
with open("enriched_user_profiles.json", "r", encoding="utf-8") as f:
    profiles = json.load(f)

# Run classification
df = pd.read_csv("translated_output.csv")
profiles = classify_user_ips_with_llm(profiles, df)

# Save result
with open("profiles_with_ip_analysis.json", "w", encoding="utf-8") as f:
    json.dump(profiles, f, ensure_ascii=False, indent=2)

In [ ]:
# Load selected user profiles
with open("profiles_with_ip_analysis.json", "r", encoding="utf-8") as f:
    selected_profiles = json.load(f)

# Load generated summaries from LLM
with open("user_summaries.json", "r", encoding="utf-8") as f:
    summaries = json.load(f)

# Inject summaries into profiles
for user, summary in summaries.items():
    if user in selected_profiles:
        selected_profiles[user]["self_summary"] = summary

# Save enriched profiles to disk
with open("user_profiles_final.json", "w", encoding="utf-8") as f:
    json.dump(selected_profiles, f, ensure_ascii=False, indent=2)

print(f"✅ Enriched profiles saved for {len(selected_profiles)} users.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pandas import Timedelta

# Load and prepare data
df = pd.read_csv('translated_output-final.csv')
df['ts'] = pd.to_datetime(df['ts'])
df.set_index('ts', inplace=True)

# Daily message counts
daily_counts = df.resample('D').size()

# Plot setup
fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(daily_counts.index, daily_counts.values, width=1.5, color='steelblue')

# Add small padding: e.g., 3 days on each side
padding = Timedelta(days=7)
ax.set_xlim(daily_counts.index.min() - padding, daily_counts.index.max() + padding)

# Major ticks every month
ax.xaxis.set_major_locator(mdates.MonthLocator())

# Formatter: show 'Jan 2021', 'Feb', 'Mar', etc.
def format_month_year(x, pos=None):
    date = mdates.num2date(x)
    return date.strftime('%b\n%Y') if date.month == 1 else date.strftime('%b')

ax.xaxis.set_major_formatter(plt.FuncFormatter(format_month_year))

# Set font size for both axes
ax.tick_params(axis='both', labelsize=13)  # or try 14 or 16 for larger

# Styling
plt.ylabel('Messages per Day', fontsize=15)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()

# Save as SVG
plt.savefig('chat_activity.eps', format='eps', facecolor='none')
plt.show()


In [ ]:
pip install matplotlib squarify

In [ ]:
import matplotlib.pyplot as plt
import squarify  # for treemap
import matplotlib as mpl

# Set the data (sample based on your screenshot)
top_messages = {
    "Error: message encrypted and cannot be decrypted": 14374,
    "hello": 7286,
    "ok": 3269,
    "yes": 1720,
    "[error: this message is encrypted, and you are unable to decrypt it": 1097,
    "here": 835,
    "Okay. Yep": 823,
    "good": 707,
    "hi": 460,
    "ok.": 427,
}

# Group rest into 'Other Frequent Messages' and 'Less Frequent Messages'
other_frequent = 11641
non_textual_messages = 6160
less_frequent = 121038  # placeholder for unseen messages (change as needed)

# Build data for treemap
labels = list(top_messages.keys()) + [
    "Other Frequent Messages",
    "Less Frequent Messages",
    "Non-Textual Messages (Links, IDs, etc.)"
]
sizes = list(top_messages.values()) + [other_frequent, less_frequent, non_textual_messages]

# Colors
cmap = mpl.cm.PuBu
colors = [cmap(i / len(sizes)) for i in range(len(sizes))]

# Plot
plt.figure(figsize=(14, 9))
squarify.plot(
    sizes=sizes,
    label=labels,
    color=colors,
    alpha=0.9,
    text_kwargs={'fontsize': 10, 'wrap': True}
)
plt.axis('off')
plt.title("Treemap of Skipped Messages in LLM Preprocessing", fontsize=15, pad=20)

# Save as EPS for LaTeX
plt.savefig("llm_skipped_messages_treemap.eps", format='eps', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

# Load CSV without converting blanks to NaN
df = pd.read_csv("victim_profiles.csv", keep_default_na=False)

# Identify columns to check (all except the first two)
columns_to_check = df.columns[2:]

# Drop rows where *all* of the remaining columns are "n/a"
filtered_df = df[~(df[columns_to_check] == "n/a").all(axis=1)]

# Save to a new final CSV
filtered_df.to_csv("victim_profiles_final.csv", index=False)

print("✅ Filtered file saved as 'victim_profiles_final.csv'")